In [34]:
import pandas as pd

df = pd.read_csv('/home/hojin/text2sql-agent/mlflow_experiments/text2sql_eval/eval/results_v3.csv')

In [35]:
df

,id,difficulty,category,user_query,generated_sql,valid_sql,exec_success,exec_accuracy,error_msg,latency_ms
0,tc_001,easy,단순조회,삼성전자 2023년 영업이익 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,True,True,NaN,3565
1,tc_002,easy,단순조회,SK하이닉스 2022년 당기순이익 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,False,False,NaN,5993
2,tc_003,easy,단순조회,현대자동차 2023년 매출액 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,True,True,NaN,4815
3,tc_004,easy,단순조회,LG전자 2023년 자산총계 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,True,True,NaN,1691
4,tc_005,easy,단순조회,카카오 2022년 부채총계 알려줘,"SELECT c.corp_name, f.bsns_year, \n MAX(...",True,True,True,NaN,2089
5,tc_006,easy,단순조회,POSCO홀딩스 2023년 자본총계 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,True,True,NaN,7690
6,tc_007,easy,단순조회,셀트리온 2023년 현금및현금성자산 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,True,True,NaN,3856
7,tc_008,easy,단순조회,KB금융 2023년 법인세비용 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,False,True,NaN,5349
8,tc_009,medium,기간비교,삼성전자 2020~2023년 영업이익 추이 알려줘,"SELECT f.bsns_year, f.amount\nFROM financial_f...",True,True,True,NaN,13033
9,tc_010,medium,기간비교,현대자동차 최근 3년 매출액 추이 알려줘,"SELECT f.bsns_year, \n MAX(CASE WHEN f.a...",True,True,True,NaN,20225


In [36]:
# 에러(exec_success가 'False')가 뜬 행만 추출
df_is_error = df.loc[df['exec_success'] ==False]
df_is_error

,id,difficulty,category,user_query,generated_sql,valid_sql,exec_success,exec_accuracy,error_msg,latency_ms
1,tc_002,easy,단순조회,SK하이닉스 2022년 당기순이익 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,False,False,NaN,5993
7,tc_008,easy,단순조회,KB금융 2023년 법인세비용 알려줘,"SELECT c.corp_name, f.bsns_year, f.amount\nFRO...",True,False,True,NaN,5349
11,tc_012,medium,기간비교,SK하이닉스 2020~2023년 자산총계 추이 알려줘,"SELECT f.bsns_year, f.amount\nFROM financial_f...",True,False,True,NaN,1699
12,tc_013,medium,기간비교,NAVER 최근 4년 영업비용 추이 알려줘,"SELECT f.bsns_year, \n MAX(CASE WHEN f.a...",True,False,True,NaN,1606
15,tc_016,medium,기간비교,신한지주 2021~2023년 법인세비용 추이 알려줘,"SELECT f.bsns_year, f.amount\nFROM financial_f...",True,False,True,NaN,10266
20,tc_021,medium,파생지표,SK하이닉스 2023년 매출총이익률 알려줘,"SELECT c.corp_name, f.bsns_year,\n ROUND(M...",True,False,True,NaN,1706


In [37]:
failed = df[df['exec_success']==False][['id','user_query','generated_sql','error_msg']]
for _, row in failed.iterrows():
    print(f"\n{'='*60}")
    print(f"ID: {row['id']}")
    print(f"Query: {row['user_query']}")
    print(f"SQL: {row['generated_sql']}")
    print(f"Error: {row['error_msg']}")


ID: tc_002
Query: SK하이닉스 2022년 당기순이익 알려줘
SQL: SELECT c.corp_name, f.bsns_year, f.amount
FROM financial_fact f
JOIN company_dim c ON f.corp_code = c.corp_code
WHERE c.corp_name LIKE '%에스케이하이닉스(주)%'
AND f.account_nm LIKE '%당기순이익%'
AND f.bsns_year = 2022
AND f.fs_div = 'CFS'
AND f.sj_div = 'IS'
Error: nan

ID: tc_008
Query: KB금융 2023년 법인세비용 알려줘
SQL: SELECT c.corp_name, f.bsns_year, f.amount
FROM financial_fact f
JOIN company_dim c ON f.corp_code = c.corp_code
WHERE c.corp_name LIKE '%(주)KB금융지주%'
AND f.account_nm LIKE '%법인세비용%'
AND f.bsns_year = 2023
AND f.fs_div = 'CFS'
AND f.sj_div = 'IS'
Error: nan

ID: tc_012
Query: SK하이닉스 2020~2023년 자산총계 추이 알려줘
SQL: SELECT f.bsns_year, f.amount
FROM financial_fact f
JOIN company_dim c ON f.corp_code = c.corp_code
WHERE c.corp_name LIKE '%SK하이닉스(주)%'
AND f.account_nm LIKE '%자산총계%'
AND f.bsns_year BETWEEN 2020 AND 2023
AND f.fs_div = 'CFS'
AND f.sj_div = 'BS'
ORDER BY f.bsns_year
Error: nan

ID: tc_013
Query: NAVER 최근 4년 영업비용 추이 알려줘
SQL: SELECT f.bsns_